# Project Big Data - Notebook Group 22; Amazon Books

## Datasets Import and Setup
In the following block we import the datasets and prepare them for further usage.

In [3]:
import warnings

warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import sklearn as skl
import pycountry
import plotly.express as px

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

CUR_DIR = os.getcwd()
BOOKS_DATA_PATH = os.path.join(CUR_DIR, "books_data/books.csv")
RATINGS_DATA_PATH = os.path.join(CUR_DIR, "books_data/ratings.csv")
USERS_DATA_PATH = os.path.join(CUR_DIR, "books_data/users.csv")

df_books = pd.read_csv(BOOKS_DATA_PATH, encoding = 'latin-1', sep = ';', quotechar='"', escapechar="\\").drop(columns = ['Image-URL-S', 'Image-URL-M', 'Image-URL-L'])
df_ratings = pd.read_csv(RATINGS_DATA_PATH, encoding = 'latin-1', sep = ';', quotechar='"')
df_users = pd.read_csv(USERS_DATA_PATH, encoding = 'latin-1', sep = ';', quotechar='"', escapechar="\\", index_col=0)

print(f"Books data    loaded {len(df_books):,} rows, {df_books.shape[1]} columns.")
print(f"Ratings data  loaded {len(df_ratings):,} rows, {df_ratings.shape[1]} columns.")
print(f"Users data    loaded {len(df_users):,} rows, {df_users.shape[1]} columns.")

df_active_countries = df_users['Location'].str.split(',').str[-1].str.strip()
unique_active_countries = df_active_countries.unique()

Books data    loaded 271,379 rows, 5 columns.
Ratings data  loaded 1,149,780 rows, 3 columns.
Users data    loaded 278,858 rows, 2 columns.


In the block below we define a list of countries that will be used further. This is a step of cleaning customers data, so we are left only with realistic and standardized countries where customers originate from. A "realistic and standardized country" is a relative term; primarily it is interpreted as the english name or abbreviation of a country's name.

We use a two-step validation method where we first obtain a list of mutual countries and teritories, present in the dataset and in the python library pycountry, storing official names of most countries in the world (METHOD 1). Then (in METHOD 2) we check the data from countries with more than 50 customers, and compare if we are missing any country from the "mutual" list. We manually add those missing countries to the "mutual" list. 

At the end we preserved about 98% of the users from the original set, organizing them in each country accordingly. The other 2% is removed due to the unstandardized nature of the addresses. Note, there are two countries with more than 50 users, that we decided not to include: "españa" due to having about 65 customers, while the majority has correctly used the standard "spain", and "yugoslavia" due to having about 180 customers, however the country no longer existing.

In [10]:
"""
METHOD 1
"""
countries = []
for country in pycountry.countries:
    countries.append(country.name.lower())

mutual = []
for mutual_country in countries:
    if mutual_country in unique_active_countries:
        mutual.append(mutual_country)

#ADD MANUALLY countries not in mutual
mutual += "usa", "russia", "iran", "vietnam", "u.a.e", "turkey", "taiwan", "syria", "venezuela", "south korea", "czech republic"
 
total_users_method1 = 0
for country in mutual:
    total_users_method1 += df_active_countries[df_active_countries == country].count()
print(f"Percentage of users kept after cleaning;            METHOD 1: {round(total_users_method1/len(df_users)*100, 2)}")

"""
METHOD 2
this is an additional validation step of the mutual list
"""
countries_counts = df_active_countries.value_counts()
counts_list = countries_counts[countries_counts>50].index.tolist()
counts_list.remove("")

total_users_method2 = 0
for country in counts_list:
    total_users_method2 += df_active_countries[df_active_countries == country].count()
print(f"Percentage of users kept after cleaning; only using METHOD 2: {round(total_users_method2/len(df_users)*100, 2)}")
print(f"Countries, having more than 50 customers, excluded from the countries list are: {set(counts_list)- set(mutual)}")

df_users['Country'] = df_users['Location'].str.split(',').str[-1].str.strip().str.lower()
df_users['Country'] = df_users['Country'].apply(lambda x: x if x in mutual else "unknown")

Percentage of users kept after cleaning;            METHOD 1: 97.92
Percentage of users kept after cleaning; only using METHOD 2: 97.38
Countries, having more than 50 customers, excluded from the countries list are: {'españa', 'yugoslavia'}


# Metrics
Below are presented basic metrics per country.

In [11]:
df_ratings_users = df_ratings.merge(df_users, on = 'User-ID')

df_mean_ratings = df_ratings_users.groupby('Country')['Book-Rating'].mean().reset_index()
df_ratings_frequency = df_ratings_users.groupby('Country')['Book-Rating'].count().reset_index()

fig = px.choropleth(
    df_mean_ratings,
    locations='Country',
    locationmode='country names',
    color='Book-Rating',
    title='Average Book Rating per Country',
    hover_data='Book-Rating',
    color_continuous_scale='bluyl'
)
fig.show()

fig = px.choropleth(
    df_ratings_frequency,
    locations='Country',
    locationmode='country names',
    color='Book-Rating',
    title='Reviews Frequency per Country',
    hover_data='Book-Rating',
    color_continuous_scale='bluyl'
)
fig.show()

#average rate per age group
